In [0]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [0]:
df = pd.read_csv("/Workspace/Users/madebypratham@gmail.com/ML-notes/data/Mental Health Dataset.csv")
df.head()



In [0]:
df.shape

In [0]:
df.isnull().sum()

In [0]:
df["self_employed"].value_counts()

In [0]:
df["Occupation"].value_counts()

In [0]:
df["Gender"].value_counts()

In [0]:
df["Country"].value_counts()

In [0]:
df["self_employed"].fillna(df["self_employed"].mode()[0], inplace = True)

In [0]:
df.isnull().sum()

In [0]:
df.duplicated().sum()

In [0]:
df.drop_duplicates(inplace = True)

In [0]:
df.duplicated().sum()

In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
encodO = OneHotEncoder(sparse_output= False)
OneHOt = ["Gender", "self_employed",]
encoded = encodO.fit_transform(df[OneHOt])
encoded = pd.DataFrame(encoded, columns= encodO.get_feature_names_out(OneHOt),
                       index= df.index)

In [0]:
df = pd.concat([df.drop(OneHOt, axis= 1), encoded], axis=1)

In [0]:
df.head()

In [0]:
LebelEn = ["family_history","treatment","Country", "Occupation", "Days_Indoors", "Growing_Stress", "Changes_Habits", "Mental_Health_History", "Mood_Swings", "Work_Interest", "Coping_Struggles",
          "Social_Weakness", "mental_health_interview","care_options"]
encodL = LabelEncoder()
for i in LebelEn:
    df[i] = encodL.fit_transform(df[i])

In [0]:
df[i]

In [0]:
df.head()

In [0]:
df["Timestamp"].isna().sum()

In [0]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format= "%m/%d/%Y %M:%S")

In [0]:
df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Days"] = df["Timestamp"].dt.day
df["Weekday"] = df["Timestamp"].dt.dayofweek

In [0]:
df.head()

In [0]:
df.shape

In [0]:
df.drop("Timestamp",axis=1)

In [0]:
# drop the original Timestampsome columns
df1 = df.drop(["Timestamp","Gender_Female","self_employed_Yes","Year","Month","Days","Weekday"], axis= 1)

corr = df1.corr()
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot= True)

In [0]:
df1.rename(columns= {
    "Gender_Male" : "Gender",
    "self_employed_No": "self_employed"
}, inplace= True)

In [0]:
# df1.rename(columns= {
#     "family_history_Yes" : "family_history",
#     "treatment_Yes": "treatment"
# }, inplace= True)

In [0]:
df1.head()

In [0]:
df1.info()

In [0]:
x = df1.drop("treatment", axis= 1)
y = df1["treatment"]

In [0]:
y

In [0]:
# Train - split test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state= 40)

In [0]:
x_train

In [0]:
x_train.shape

In [0]:
# Scale the data
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
x_train = scale.fit_transform(x_train)

In [0]:
x_train

In [0]:
# fit the model
from sklearn.naive_bayes import GaussianNB, BernoulliNB
model = GaussianNB(var_smoothing= 1e-12)
model.fit(x_train, y_train)

In [0]:
model1 = BernoulliNB()
model1.fit(x_train, y_train)

In [0]:
model.score(x_train, y_train)

In [0]:
model.score(x_test, y_test)

In [0]:
model1.score(x_train, y_train)

### Hyperparameter Tunning by using Grid Search CV

In [0]:
from sklearn.model_selection import GridSearchCV
classifer = GridSearchCV((model), {
    "var_smoothing": [1e-12, 1e-10, 1e-9, 1e-8, 1e-7],
    "priors": [None]   # Usually keep None (learn from data)
}, cv= 5,scoring= "f1" ,return_train_score= False)
classifer.fit(x_train, y_train)

In [0]:
# cross check it's accuracy by confusion matrics
from sklearn.metrics import confusion_matrix, classification_report
y_pred = model.predict(x_test)
confusion_matrix(y_test, y_pred)

In [0]:
print(classification_report(y_test, y_pred))